# Análisis estadístico de la tesis
## Adopción tecnológica en el sector financiero costarricense

**Autor:** José Andrés Vargas Vargas — MBA ULACIT — Investigación Empresarial Aplicada (28-0017)

**Objetivo del notebook:** ejecutar de manera reproducible el pipeline completo de análisis cuantitativo de la encuesta de 15 ítems, contrastar las hipótesis H1–H6 mediante correlaciones, generar las visualizaciones del Capítulo 4 y exportar tablas listas para integrar en el documento APA 7.

**Bibliotecas:** pandas, numpy, scipy.stats, pingouin, matplotlib, seaborn.

**Datos esperados:** archivo `encuesta_respuestas.csv` con la estructura definida en `encuesta_template.csv`.

**Cómo correrlo:**
1. `pip install -r requirements.txt`
2. Coloque su archivo de respuestas como `encuesta_respuestas.csv` en la misma carpeta.
3. Ejecute todas las celdas (`Cell → Run All`).

## 0. Configuración del entorno

In [ ]:
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

try:
    import pingouin as pg
    HAS_PINGOUIN = True
except ImportError:
    HAS_PINGOUIN = False
    print('pingouin no instalado: el alfa de Cronbach se calculará manualmente.')

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

sns.set_theme(style='whitegrid', context='paper', font_scale=1.1)
plt.rcParams['figure.dpi'] = 110

print(f'Python {sys.version.split()[0]}')
print(f'pandas {pd.__version__} | numpy {np.__version__} | scipy {stats.__name__}')


## 1. Carga de datos

Si todavía no tiene su CSV con respuestas reales, el notebook cargará la plantilla `encuesta_template.csv` que sirve como ejemplo.

In [ ]:
from pathlib import Path

data_path = Path('encuesta_respuestas.csv')
if not data_path.exists():
    data_path = Path('encuesta_template.csv')
    print('Aviso: usando plantilla de ejemplo (encuesta_template.csv).')

df = pd.read_csv(data_path)
print(f'Total respuestas cargadas: {len(df)}')
df.head()


## 2. Validación de datos

Verificación de tipos, valores faltantes y rangos válidos para los ítems Likert (1–5).

In [ ]:
likert_cols = [c for c in df.columns if c.startswith('Q') and c not in ['Q1_tipo_entidad', 'Q2_area_funcional', 'Q3_antiguedad']]
demo_cols = ['Q1_tipo_entidad', 'Q2_area_funcional', 'Q3_antiguedad']

print('=== Valores faltantes por columna ===')
print(df.isna().sum())

print('\n=== Rango de ítems Likert ===')
print(df[likert_cols].agg(['min', 'max']).T)

fuera_de_rango = ((df[likert_cols] < 1) | (df[likert_cols] > 5)).sum().sum()
print(f'\nValores fuera del rango 1-5: {fuera_de_rango}')


## 3. Recodificación

Los ítems Q6 (lentitud) y Q12 (cuellos de botella) están fraseados en sentido contrario y deben recodificarse para que valores altos signifiquen mayor presencia del factor.

In [ ]:
df['Q6_v2_lentitud_estructural_a_inv'] = 6 - df['Q6_v2_lentitud_estructural_a']
df['Q12_v5_cuellos_botella_a_dir'] = df['Q12_v5_cuellos_botella_a']  # ya está en sentido directo

constructos = {
    'V1_adopcion_temprana': ['Q4_v1_adopcion_temprana_a', 'Q5_v1_adopcion_temprana_b'],
    'V2_lentitud_estructural': ['Q6_v2_lentitud_estructural_a_inv', 'Q7_v2_lentitud_estructural_b'],
    'V3_devops_agile': ['Q8_v3_devops_a', 'Q9_v3_devops_b'],
    'V4_arquitectura': ['Q10_v4_arquitectura_a', 'Q11_v4_arquitectura_b'],
    'V5_cuellos_botella': ['Q12_v5_cuellos_botella_a_dir', 'Q13_v5_cuellos_botella_b'],
    'V6_talento_tecnico': ['Q14_v6_talento'],
    'VD_avance_digital': ['Q15_vd_avance_digital'],
}

for nombre, items in constructos.items():
    df[nombre] = df[items].mean(axis=1)

df[list(constructos.keys())].head()


## 4. Estadísticos descriptivos

Generación de la tabla de medias, desviaciones estándar, cuartiles y rango por dimensión (Tabla 4 del Capítulo 4).

In [ ]:
tabla_descriptivos = df[list(constructos.keys())].describe().T
tabla_descriptivos = tabla_descriptivos.round(2)
tabla_descriptivos.to_csv('tabla_descriptivos.csv')
tabla_descriptivos


In [ ]:
# Distribución de los demográficos
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, demo_cols):
    df[col].value_counts().plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.set_ylabel('Frecuencia')
    ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('fig_demograficos.png', dpi=200, bbox_inches='tight')
plt.show()


## 5. Confiabilidad (alfa de Cronbach)

Por dimensión, considerando aceptable α ≥ 0.70 (Tavakol & Dennick, 2011).

In [ ]:
def cronbach_alpha(items_df: pd.DataFrame) -> float:
    """Cálculo manual del alfa de Cronbach para casos sin pingouin."""
    items = items_df.dropna()
    k = items.shape[1]
    if k < 2:
        return np.nan
    var_items = items.var(axis=0, ddof=1).sum()
    var_total = items.sum(axis=1).var(ddof=1)
    return (k / (k - 1)) * (1 - var_items / var_total)

rows = []
for nombre, items in constructos.items():
    if len(items) >= 2:
        if HAS_PINGOUIN:
            alpha, _ = pg.cronbach_alpha(data=df[items])
        else:
            alpha = cronbach_alpha(df[items])
        rows.append({'Dimensión': nombre, 'k_ítems': len(items), 'Cronbach_alpha': round(alpha, 3)})
    else:
        rows.append({'Dimensión': nombre, 'k_ítems': len(items), 'Cronbach_alpha': 'N/A (1 ítem)'})

tabla_alpha = pd.DataFrame(rows)
tabla_alpha.to_csv('tabla_cronbach.csv', index=False)
tabla_alpha


## 6. Contraste de hipótesis (H1–H6)

Cada hipótesis se contrasta mediante correlación bivariada entre la variable independiente y la variable dependiente (`VD_avance_digital`).

Pasos:
1. Verificación del supuesto de normalidad (Shapiro-Wilk).
2. Si ambas variables son normales → Pearson; si no → Spearman.
3. Reporte de coeficiente, *p*-valor y decisión sobre la hipótesis.

In [ ]:
ALPHA = 0.05
vd = df['VD_avance_digital']

hipotesis = {
    'H1': ('V1_adopcion_temprana', 'positiva'),
    'H2': ('V2_lentitud_estructural', 'negativa'),
    'H3': ('V3_devops_agile', 'positiva'),
    'H4': ('V4_arquitectura', 'positiva'),
    'H5': ('V5_cuellos_botella', 'negativa'),
    'H6': ('V6_talento_tecnico', 'positiva'),
}

filas = []
for h, (var, direccion_esperada) in hipotesis.items():
    x = df[var]
    p_norm_x = stats.shapiro(x).pvalue if len(x) >= 3 else np.nan
    p_norm_y = stats.shapiro(vd).pvalue if len(vd) >= 3 else np.nan
    metodo = 'Pearson' if (p_norm_x > 0.05 and p_norm_y > 0.05) else 'Spearman'
    if metodo == 'Pearson':
        r, p = stats.pearsonr(x, vd)
    else:
        r, p = stats.spearmanr(x, vd)
    direccion_observada = 'positiva' if r > 0 else 'negativa'
    cumple = (p < ALPHA) and (direccion_observada == direccion_esperada)
    filas.append({
        'Hipótesis': h,
        'Variable': var,
        'Método': metodo,
        'r': round(r, 3),
        'p-valor': round(p, 4),
        'Dirección esperada': direccion_esperada,
        'Dirección observada': direccion_observada,
        'Decisión': 'Se acepta H' if cumple else 'No se acepta H',
    })

tabla_hipotesis = pd.DataFrame(filas)
tabla_hipotesis.to_csv('tabla_hipotesis.csv', index=False)
tabla_hipotesis


## 7. Comparaciones entre grupos

Se compara la VD entre tipos de entidad y áreas funcionales mediante ANOVA.

In [ ]:
def anova_grupos(df, var_grupo, var_dependiente):
    grupos = [g[var_dependiente].dropna().values for _, g in df.groupby(var_grupo)]
    grupos = [g for g in grupos if len(g) >= 2]
    if len(grupos) < 2:
        return None
    f, p = stats.f_oneway(*grupos)
    return {'variable_grupo': var_grupo, 'F': round(f, 3), 'p-valor': round(p, 4)}

comparaciones = []
for col in demo_cols:
    r = anova_grupos(df, col, 'VD_avance_digital')
    if r:
        comparaciones.append(r)

tabla_comparaciones = pd.DataFrame(comparaciones)
tabla_comparaciones.to_csv('tabla_comparaciones.csv', index=False)
tabla_comparaciones


## 8. Visualizaciones para el Capítulo 4

### 8.1 Ranking de factores por correlación con la variable dependiente

In [ ]:
ranking = tabla_hipotesis[['Variable', 'r']].copy()
ranking['|r|'] = ranking['r'].abs()
ranking = ranking.sort_values('|r|', ascending=True)

colors = ['#2c7fb8' if r > 0 else '#d7301f' for r in ranking['r']]

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(ranking['Variable'], ranking['r'], color=colors, edgecolor='black')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Coeficiente de correlación con VD_avance_digital')
ax.set_title('Ranking de factores por correlación')
ax.set_xlim(-1, 1)
for i, (var, r) in enumerate(zip(ranking['Variable'], ranking['r'])):
    ax.text(r + (0.02 if r >= 0 else -0.02), i, f'{r:.2f}', va='center',
            ha='left' if r >= 0 else 'right')
plt.tight_layout()
plt.savefig('fig_ranking_factores.png', dpi=200, bbox_inches='tight')
plt.show()


### 8.2 Mapa de calor de la matriz de correlaciones entre constructos

In [ ]:
matriz_cor = df[list(constructos.keys())].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(matriz_cor, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title('Matriz de correlaciones entre constructos')
plt.tight_layout()
plt.savefig('fig_heatmap_correlaciones.png', dpi=200, bbox_inches='tight')
plt.show()


### 8.3 Distribución de la variable dependiente

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.histplot(df['VD_avance_digital'], bins=10, kde=True, ax=ax,
             color='steelblue', edgecolor='black')
ax.set_xlabel('Percepción de avance digital (1-5)')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución de la variable dependiente')
plt.tight_layout()
plt.savefig('fig_distribucion_vd.png', dpi=200, bbox_inches='tight')
plt.show()


## 9. Reporte ejecutivo de hallazgos

Síntesis automática que se puede pegar directamente en el Capítulo 4.

In [ ]:
n = len(df)
vd_media = df['VD_avance_digital'].mean()
vd_sd = df['VD_avance_digital'].std()

print(f'Tamaño de la muestra: n = {n}')
print(f'Percepción de avance digital: M = {vd_media:.2f}, SD = {vd_sd:.2f}\n')
print('=== Resultados del contraste de hipótesis ===')
for _, row in tabla_hipotesis.iterrows():
    print(f"{row['Hipótesis']} ({row['Variable']}): r = {row['r']}, p = {row['p-valor']} → {row['Decisión']}")

print('\n=== Ranking de factores por |r| ===')
for _, row in ranking.sort_values('|r|', ascending=False).iterrows():
    print(f"  {row['Variable']}: r = {row['r']}")


## 10. Exportación final

Todas las tablas se exportan como CSV en el directorio de trabajo. Las figuras se exportan como PNG en alta resolución (200 dpi). Estos artefactos se incluyen en el documento APA 7.

In [ ]:
import os
artefactos = sorted(
    f for f in os.listdir('.')
    if f.startswith(('tabla_', 'fig_'))
)
print('Artefactos generados:')
for a in artefactos:
    print(f'  • {a}')
